Imports, WRDS connection

In [5]:
import wrds
import pandas as pd
import numpy as np
import cvxpy as cp
import os

# Connecting WRDS
#db = wrds.Connection()
print("Connected to WRDS")

Connected to WRDS


**Variables and conventions:**

**Fixed Quantities**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $H$                           | combined factor-risk + estimation-uncertainty matrix | $m \times m$ matrix |
| $Q$                           | eigenvectors of $H$ | $m \times m$ matrix |
| $\theta$ (theta)              | eigenvalues of $H$ | $m \times 1$ vector |
| $\theta_{max}$ (theta_max)    | max eigenvalue of $H$ | scalar (float) |
| $A$                           | rotated factor exposure map ($Q^T H^{frac{1}{2}}G^{frac{1}{2}}V_0) | $m \times n$ matrix |

**Per-asset / uncertainty-set parameters**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $\mu_0$ (mu0)                 | point estimate of mean returns | $n \times 1$ vector |
| $\alpha$ (alpha)              | lower bound on required portfolio return | scalar (float) |
| $\gamma$ (gamma)              | box uncertainty radius on mean returns | $n \times 1$ vector |
| $\rho$ (rho)                  | ellipsoidal uncertainty radius on factor loadings | $n \times 1$ vector |
| $D$                           | covariance matrix, where $d_i$ lies in interval $[\underline{d}_i, \bar{d}_i], i = 1, ... , n$ | $n \times n$ matrix |
| $\bar{D}$                     | Diagonal matrix of box uncertainty upper bound values of $d$ on idiosyncratic variance | $n \times n$ matrix |
| $\bar{d}$                     | box uncertainty upper bound on idiosyncratic variance | $n \times 1$ vector |

**Dimensions**
| Variable | Meaning |
| -------- | ------- |
| $n$      | number of assets  |
| $m$      | number of factors |

**Decision Variables**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $\phi$ (phi)      | portfolio weights      | n x 1 vector |
| $\lambda$ (lamda) | factor variance budget | scalar (float) |
| $\delta$ (delta)  | idiosyncratic variance budget | scalar (float) |
| $\zeta$ (zeta)    | absolute value bound on phi (for C3, C4) | n x 1 vector |
| $\sigma$ (sigma)  | S-procedure multiplier | scalar (float) |
| $\tau$ (tau)            | auxiliary variable in C6/C8 | scalar (float) |
| $t$            | per-factor auxiliary variable, length m (C6/C9) | $m \times 1$ vector |






Helpers

In [ ]:
def compute_matrix_power(A: np.ndarray, n: float) -> np.ndarray:
    """
    Compute A^n, assuming A is positive semidefinite
    """
    # 1. Compute eigenvalues (w) and eigenvectors (v)
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    eigenvalues = np.clip(eigenvalues, 0, None) # for numerical stability

    # 2. Raise eigenvalues to the power of n
    # (Avoid division by zero by ensuring eigenvalues are positive)
    inv_sqrt_eigenvalues = eigenvalues ** n

    # 3. Reconstruct the matrix: V * diag(λ^n) * V.T
    A_n = eigenvectors @ np.diag(inv_sqrt_eigenvalues) @ eigenvectors.T

    return A_n

def compute_fixed_quantities(G: np.ndarray, F: np.ndarray, V0: np.ndarray) -> tuple:
    """
    Compute H, and its eigendecomposition, and A
    """

    #Ensure G is stable
    if np.linalg.cond(G) > 1e10:
        G = G + 1e-8 * np.eye(G.shape[0])

    G_minus_half = compute_matrix_power(G, -0.5)
    H = G_minus_half @ F @ G_minus_half

    eigenvalues, eigenvectors = np.linalg.eigh(H)
    Q = eigenvectors
    Lambda = np.diag(eigenvalues)
    H_decomposed = Q @ Lambda @ Q.T

    theta = eigenvalues
    theta_max = theta.max()

    H_sqrt = compute_matrix_power(H, 0.5)
    G_sqrt = compute_matrix_power(G, 0.5)
    A = Q.T @ H_sqrt @ G_sqrt @ V0

    """
    For testing:
    m = F.shape[0]
        n = V0.shape[1]
    
    
    if not np.allclose(H, H.T) or not A.shape == (m,n) or not np.allclose(H, H_decomposed):
                print("Computational error")
                return ()
    """
    

    return (H, Q, Lambda, theta, theta_max, A)

3. Solver

In [ ]:
def solve_gi_socp(H: np.ndarray, Q: np.ndarray, theta: np.ndarray, 
                  theta_max: float, A: np.ndarray, mu0: np.ndarray, 
                  gamma: np.ndarray, rho: np.ndarray, d_bar: np.ndarray, alpha: float) -> dict:

    #Find dimensions
    n = d_bar.shape[0]
    m = H.shape[0]
    
    #defining decision and objective variables
    phi = cp.Variable(n)
    lam = cp.Variable()
    delta = cp.Variable()
    zeta  = cp.Variable(n)
    sigma = cp.Variable()
    tau = cp.Variable()
    t = cp.Variable(m)
    n_ones = np.ones(n)

    #Construct SOC constraints
    #C1:
    sqrt_d_bar = np.sqrt(d_bar)
    v1 = cp.multiply(2*sqrt_d_bar, phi)
    soc_arg1 = cp.hstack([v1, 1 - delta])
    C1 = cp.SOC(1 + delta, soc_arg1)

    #C8:
    r = rho.T @ zeta
    soc_arg8 = cp.hstack([2 * r, sigma - tau])
    C8 = cp.SOC(sigma + tau, soc_arg8)

    #C9:
    w = A @ phi
    C9_list = [cp.SOC(1 - sigma * theta[i] + t[i], cp.hstack([2 * w[i], 1 - sigma * theta[i] - t[i]])) for i in range(m)]

    #Construct linear constraint list:
    C2 = mu0 @ phi - gamma @ phi >= alpha       #C2
    C3 = phi <= zeta                            #C3
    C4 = -phi <= zeta                           #C4
    C5 = n_ones @ phi == 1                      #C5
    C6 = tau + n_ones @ t <= lam                #C6
    C7 = sigma <= 1.0/theta_max                 #C7

    linear_constraints = [C2, C3, C4, C5, C6, C7]

    #Bundle SOC constraints with linear constraints
    all_constraints = [C1, C8] + C9_list + linear_constraints 

    #Define objective
    problem = cp.Problem(cp.Minimize(lam + delta), all_constraints)
    try:
        problem.solve(solver=cp.MOSEK)
    except (cp.error.SolverError):
        problem.solve(solver=cp.ECOS)  # Backup solver
        

    if not problem.status == "optimal":
        raise ValueError("Solver solution was not optimal, try again with alpha = 0")

    values_dict = {
        "phi": phi.value,
        "lam": lam.value,
        "delta": delta.value,
        "sigma": sigma.value,
        "zeta": zeta.value,
        "t": t.value,
        "tau": tau.value,
        "problem.value": problem.value,
        "mu1": C1.dual_value,
        "mu2": C2.dual_value,
        "mu3": C3.dual_value,
        "mu4": C4.dual_value,
        "mu5": C5.dual_value,
        "mu6": C6.dual_value,
        "mu7": C7.dual_value,
        "mu8": C8.dual_value,
    }

    values_dict["mu9"] = np.array([c.dual_value for c in C9_list])

    return values_dict

